In [ ]:
from pathlib import Path
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/NGFS/").iterdir())
    ]
)

In [ ]:
# rename scenarios for clear reference to ENGAGE
df.rename(
    scenario=dict([(i, "NGFS Phase 5-" + i) for i in df.scenario]),
    inplace=True,
)

In [ ]:
df.rename(
    region=dict([(i, i.replace(" NGFS", "")) for i in df.filter(region="GCAM*").region]),
    inplace=True,
)

In [ ]:
df

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
df.rename(
    region={
        "GCAM 6.0|Australia_NZ": "GCAM 6.0|Australia and New Zealand",
        "MESSAGEix-GLOBIOM 2.0-R12|Rest Centrally Planned Asia": "MESSAGEix-GLOBIOM 2.0-R12|Rest of Centrally Planned Asia",
        "MESSAGEix-GLOBIOM 2.0-R12|Sub-saharan Africa": "MESSAGEix-GLOBIOM 2.0-R12|Sub-Saharan Africa",
        "REMIND-MAgPIE 3.3-4.8|Canada, NZ, Australia": "REMIND-MAgPIE 3.3-4.8|Canada, Australia, New Zealand",
        "REMIND-MAgPIE 3.3-4.8|China": "REMIND-MAgPIE 3.3-4.8|China and Taiwan",
        "REMIND-MAgPIE 3.3-4.8|Countries from the Reforming Economies of the Former Soviet Union": "REMIND-MAgPIE 3.3-4.8|Russia and Reforming Economies",
        "REMIND-MAgPIE 3.3-4.8|Middle East, North Africa, Central Asia": "REMIND-MAgPIE 3.3-4.8|Middle East and North Africa",
        "REMIND-MAgPIE 3.3-4.8|Sub-saharan Africa": "REMIND-MAgPIE 3.3-4.8|Sub-Saharan Africa",
    },
    inplace=True,
)

In [ ]:
rename_aggregator = nomenclature.processor.Aggregator.from_file(
    "../../common-definitions/legacy/ngfs5-variable-transfer.yaml"
)

In [ ]:
df = rename_aggregator.apply(df)

In [ ]:
project = ["ngfs5", "navigate"]
legacy_mapping = {}

for _project in project:
    for code, attrs in definition.variable.items():
        if _project in attrs.extra_attributes:
            legacy_mapping[attrs.__getattr__(_project)] = code
    
    df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "km3/yr": "million m3/yr",
        "US$2010/GJ": "USD_2010/GJ",
    },
    inplace=True,
)

In [ ]:
df.rename(
    variable=dict(
        [
            (
                i,
                i.replace("Residential and Commercial|Commercial", "Commercial").replace("Residential and Commercial|Residential", "Residential")
            )
            for i in df.filter(variable="*Residential and Commercial|*").variable
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    variable=dict(
        [
            (
                i,
                (
                    i.replace("Non-ferrous metals", "Non-Ferrous Metals")
                    .replace("High value chemicals", "High-Value Chemicals")
                    .replace("Industry|Cement", "Industry|Non-Metallic Minerals|Cement")
                    .replace("Industry|Steel", "Industry|Iron and Steel")
                )
            )
            for i in df.variable
        ]
    ),
    inplace=True,
)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Storage",
        "*including medium chronic physical risk damage estimate",
        "*Average 2016-2030",
        "Post-processed*",
        "GDP|MER|Counterfactual without damage",
        "GDP|PPP|Counterfactual without damage",
        "Investment|Energy Supply|CO2 Transport and Storage",
        "Forcing",
        "Carbon Sequestration|CCS|Fossil|Energy|Supply|Liquids",
        "Agricultural Production|Energy", ## this is a duplicate
        "Secondary Energy",
        "Revenue|Government|Tax|Carbon*",
        "Agricultural Production|Energy|Residues",
        "Agricultural Production|Non-Energy",
        "Agricultural Production|Non-Energy|Crops",
        "Agricultural Production|Non-Energy|Livestock",
        "Price|Carbon|Demand|Industry",
        "Price|Carbon|Demand|Residential and Commercial",
        "Price|Carbon|Demand|Transportation",
        "Price|Carbon|Supply",
        "Price|*|Index",
        "Water Withdrawal|Irrigation", ## there is confusion about the unit
    ],
    keep=False,
    inplace=True
)

In [ ]:
df_2005 = df.filter(unit="Index (2005 = 1)").filter(model="MESS*", keep=False)

In [ ]:
df_2020 = pyam.concat(
    [
        df_2005.divide(y, 2020, str(y), axis="year").rename(unit={"": "Index (2020 = 1)"})
        for y in df_2005.year
    ]
)

In [ ]:
df = df.filter(unit="Index (2005 = 1)", keep=False).append(df_2020)

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
df._data.loc[(
    "MESSAGEix-GLOBIOM 2.0-M-R12-NGFS",
    "NGFS Phase 5-Delayed Transition",
    "MESSAGEix-GLOBIOM 2.0-R12|Rest of Centrally Planned Asia", "Carbon Capture|Energy|Supply|Fossil", "Mt CO2/yr", 2060)
] = 0

In [ ]:
definition.validate(df)

In [ ]:
region_processor = nomenclature.RegionProcessor.from_directory(
    "../../common-definitions/mappings/", definition,
)

In [ ]:
df_rx = region_processor.apply(region_processor.revert(df.copy()))

In [ ]:
df.append(df_rx.filter(region=["*(R9)", "*(R10)"]), inplace=True)

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
df_public = pyam.read_ixmp4(
    platform,
    scenario="NGFS Phase 5*",
    region="World",
    year=2020,
)

In [ ]:
[i for i in df_public.variable if i not in df.variable]

In [ ]:
new_variables = [i for i in df.variable if i not in df_public.variable]
new_variables

In [ ]:
for model, scenario in df.index:
    run = platform.runs.get(model=model, scenario=scenario)
    _df = df.filter(model=model, scenario=scenario, year=range(2010, 2101), variable=new_variables)
    with run.transact("Add more variables"):
        run.iamc.add(_df.data)
    print(f"Imported {model} - {scenario}")